# Tiny RNN Weather Forecast 🌦️
## Can an RNN learn the seasons and predict tomorrow's temperature?

In this notebook we build a very small weather forecasting AI.

The idea is simple:

```text
temperature for the last 14 days
             ↓
            RNN
             ↓
   tomorrow's temperature
```

We will create our own weather data, so we know exactly what pattern the AI is trying to learn.


# 1. Download the weather dataset

Our dataset contains **three years of daily temperature**. It has a clear seasonal pattern: colder winters, warmer summers, plus normal short-term weather variation.

The CSV file is stored with this GitHub project. We download it just like we would download a small real-world dataset.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Download the dataset from our GitHub repository.
DATA_URL = 'https://raw.githubusercontent.com/dujing82-blip/tiny-rnn-weather-forecast/main/weather_temperature.csv'
data = pd.read_csv(DATA_URL)

# Take a quick look at the table.
print(data.head())
print('\nNumber of days:', len(data))


## Look at the weather

Before building AI, **always look at your data**.


In [ ]:
# Put the two CSV columns into simple NumPy arrays.
days = data['day'].to_numpy()
temperature = data['temperature_f'].to_numpy(dtype=np.float32)

plt.figure(figsize=(14,5))
plt.plot(days, temperature)
plt.xlabel('Day')
plt.ylabel('Temperature (°F)')
plt.title('Three Years of Daily Temperature')
plt.grid(alpha=0.25)
plt.show()


### What should you notice?

The temperature is not a collection of independent numbers. It is a **sequence**.

Yesterday tells us something about today. The last several days tell us whether the weather is warming or cooling. The repeating yearly pattern tells us about the seasons.

That is exactly the kind of problem an **RNN — Recurrent Neural Network —** is designed for.


# 2. Turn the weather into sequences

We will give the RNN the previous **14 days** and ask it to predict **day 15**.

Example:

```text
INPUT X                          TARGET y

Day 1   52°F  ┐
Day 2   53°F  │
Day 3   51°F  │
 ...          ├──→ RNN ──→     Day 15: 58°F
Day 13  56°F  │
Day 14  57°F  ┘
```

Then we slide the window forward by one day and make another training example.


In [ ]:
SEQUENCE_LENGTH = 14

X = []   # The 14-day input sequences
y = []   # The next-day answers

for i in range(len(temperature) - SEQUENCE_LENGTH):
    # Example: temperatures from day 0 through day 13
    X.append(temperature[i : i + SEQUENCE_LENGTH])

    # Answer: temperature on day 14
    y.append(temperature[i + SEQUENCE_LENGTH])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFirst input sequence:')
print(np.round(X[0], 1))
print('Correct next temperature:', round(float(y[0]), 1), '°F')


# 3. Training data and test data

We train on the **first two years** and test on the **third year**.

This is important for time-series data: we should not randomly mix the future into the past.


In [ ]:
# The first two years are approximately the first 730 sequence targets.
split = 365 * 2 - SEQUENCE_LENGTH

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

print('Training examples:', len(X_train))
print('Testing examples:', len(X_test))


# 4. Normalize the temperatures

Neural networks usually learn more easily when numbers are on a small, centered scale.

We calculate the mean and standard deviation using **training data only**.


In [ ]:
mean_temp = np.mean(X_train)
std_temp = np.std(X_train)

X_train_scaled = (X_train - mean_temp) / std_temp
y_train_scaled = (y_train - mean_temp) / std_temp

X_test_scaled = (X_test - mean_temp) / std_temp
y_test_scaled = (y_test - mean_temp) / std_temp

# An RNN expects: [examples, time steps, features]
# We have ONE feature at each time step: temperature.
# So (716, 14) becomes (716, 14, 1).
X_train_scaled = X_train_scaled[..., np.newaxis]
X_test_scaled = X_test_scaled[..., np.newaxis]

print('RNN input shape:', X_train_scaled.shape)


# 5. Build the RNN

This is the entire neural network:

```text
14 temperatures
       ↓
   SimpleRNN
       ↓
    Dense(1)
       ↓
tomorrow's temperature
```

The RNN reads the sequence **one day at a time** and carries a hidden memory from one time step to the next.


In [ ]:
model = keras.Sequential([

    # The RNN reads 14 time steps.
    # Each time step contains ONE number: temperature.
    layers.Input(shape=(SEQUENCE_LENGTH, 1)),

    # 32 memory units are enough for this small classroom example.
    layers.SimpleRNN(32, activation='tanh'),

    # Produce ONE number: tomorrow's predicted temperature.
    layers.Dense(1)
])

model.summary()


# 6. Train the RNN

Temperature prediction is a **regression** problem: the answer is a number, not a category.

We use Keras's built-in **Mean Squared Error (MSE)** loss.


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.MeanSquaredError()
)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    shuffle=False,       # Keep time order easy to reason about.
    verbose=1
)


## Did the model learn?

The loss should generally decrease as training continues.


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('RNN Learning Curve')
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 7. Predict the third year

The network predicts normalized numbers, so we convert them back to °F.


In [ ]:
predicted_scaled = model.predict(X_test_scaled, verbose=0).flatten()

# Convert normalized predictions back into Fahrenheit.
predicted_temp = predicted_scaled * std_temp + mean_temp

# Compare a few predictions with the real answers.
for i in range(10):
    print(f'Real: {y_test[i]:5.1f}°F    Predicted: {predicted_temp[i]:5.1f}°F')


## The most important graph

Blue represents the real third-year weather. Orange represents what the RNN predicted one day ahead.


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(y_test, label='Real temperature')
plt.plot(predicted_temp, label='RNN prediction')
plt.xlabel('Day in test year')
plt.ylabel('Temperature (°F)')
plt.title('Real Weather vs. RNN One-Day Forecast')
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 8. How accurate is it?

MAE means **Mean Absolute Error**. It answers a very intuitive question:

> On average, how many degrees is our prediction off?


In [ ]:
mae = np.mean(np.abs(y_test - predicted_temp))
print(f'Mean Absolute Error: {mae:.2f} °F')


# 9. Ask the RNN to predict tomorrow

Give it any 14 consecutive temperatures. The model uses that sequence to predict the next one.


In [ ]:
def predict_tomorrow(last_14_days):
    # Convert the list to a NumPy array.
    sequence = np.array(last_14_days, dtype=np.float32)

    if len(sequence) != 14:
        print('Please give exactly 14 temperatures.')
        return

    # Normalize using the SAME numbers used during training.
    sequence = (sequence - mean_temp) / std_temp

    # RNN input shape must be: [examples, time steps, features].
    sequence = sequence.reshape(1, 14, 1)

    # Ask the RNN for its prediction.
    prediction_scaled = model.predict(sequence, verbose=0)[0,0]

    # Convert back to Fahrenheit.
    prediction = prediction_scaled * std_temp + mean_temp

    print(f'Predicted temperature tomorrow: {prediction:.1f} °F')

# Example: two weeks of gradually warming weather.
my_weather = [52, 53, 51, 54, 55, 56, 55, 57, 58, 57, 59, 60, 61, 60]
predict_tomorrow(my_weather)


# 10. What did the RNN actually learn?

The key idea is **memory across a sequence**.

```text
52° → 53° → 51° → 54° → ... → 60°
 ↓      ↓      ↓      ↓             ↓
RNN →  RNN →  RNN →  RNN → ... →  RNN
  \_____ hidden memory travels ______/
                    ↓
             tomorrow: ? °F
```

A regular Dense network sees numbers as a fixed group of inputs. An RNN is designed to read them **in order**.

### Three ideas to remember

1. **Sequence** — order matters.
2. **Hidden state** — the RNN carries information from earlier time steps.
3. **Prediction** — after reading the sequence, the RNN uses its memory to predict what comes next.

This is the foundation for more advanced sequence models such as **LSTM** and **GRU**.
